# Lesson 01 - Introduction to AI Agents

Welcome to the first lesson in the **AI Agents for Beginners** course!

An **AI agent** is a program that uses a large language model (LLM) as its reasoning engine and can take *actions* in the real world — calling APIs, querying databases, or running code — to accomplish a goal on behalf of a user.

In this notebook you will build your first agent: a **Travel Agent** that recommends vacation destinations. Along the way you will learn how to:

1. Connect to Azure AI Foundry Agent Service using the **Microsoft Agent Framework**.
2. Give the agent a **tool** — a plain Python function it can call.
3. Run the agent and inspect its response.
4. Stream the agent's response token-by-token.

## Setup

Before running this notebook, make sure you have:

1. **An Azure AI Foundry project** with a deployed chat model (e.g. `gpt-4o-mini`).
2. **Logged in with the Azure CLI** — run `az login` in your terminal.
3. **Set the required environment variables:**
   - `AZURE_AI_PROJECT_ENDPOINT` — your Azure AI Foundry project endpoint.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — the name of your deployed model.

The cell below installs the Python packages you need.

In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [3]:
# ============================================================
# CÁCH CŨ (deprecated — giữ lại để tham khảo, KHÔNG chạy)
# ============================================================
# import logging
# logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)
#
# import os
# import asyncio
# from typing import Annotated
#
# from agent_framework import tool
# from agent_framework.azure import AzureAIProjectAgentProvider  # ← đã bị xoá
# from azure.identity import AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# ============================================================

# ============================================================
# CÁCH MỚI (agent-framework >= 1.0.0, 2026) - của PH - phải là model gpt-4o-mini mới được
# ============================================================
#import os
#from dotenv import load_dotenv
#from agent_framework import tool
#from agent_framework.foundry import FoundryChatClient  # ← dùng foundry thay vì azure
#from azure.identity import DefaultAzureCredential

#load_dotenv()  # đọc biến môi trường từ file .env

#print(f"   FOUNDRY_PROJECT_ENDPOINT: {os.getenv('FOUNDRY_PROJECT_ENDPOINT')}")
#print(f"   FOUNDRY_MODEL: {os.getenv('FOUNDRY_MODEL')}")

#credential = DefaultAzureCredential()

import subprocess
import sys

print("Installing required packages...")
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "agent-framework",
    "agent-framework-foundry",
    "azure-identity",
])
print("✓ Packages installed successfully!")
print("\n⚠️  Now restart the Jupyter kernel:")
print("  - In VS Code: Click the 'Restart Kernel' button (or Ctrl+Shift+P > Restart Kernel)")
print("  - Then run the next cells")

Installing required packages...
✓ Packages installed successfully!

⚠️  Now restart the Jupyter kernel:
  - In VS Code: Click the 'Restart Kernel' button (or Ctrl+Shift+P > Restart Kernel)
  - Then run the next cells


## Creating Your First Agent

An agent needs two things:  
**Old**
- **Instructions** that tell it *who it is* and *how to behave* (a system prompt).
- **Tools** — Python functions decorated with `@tool` that the agent can call to retrieve information or perform actions.

**New**
- **A Foundry Chat Client** — FoundryChatClient with your project endpoint and model deployment.
- **Tool functions** — decorated with @tool to expose them to the agent.
- **An Agent wrapper** — created via chat_client.as_agent(...).
Then run with await agent.run(...) and read result.text.

If your deployment does not support tool-calling parameters, the code cell will automatically fallback to a no-tool mode so you can still complete the lesson.

Below we define a simple tool that returns a list of popular vacation destinations. The agent will use this tool when a user asks for travel recommendations.


In [1]:
from typing import List

# Import the tool decorator
try:
    from agent_framework import tool
except ImportError:
    print("✗ Could not import tool decorator")
    raise

@tool(approval_mode="never_require")
def get_destinations() -> List[str]:
    """Return a curated list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

print("✓ get_destinations tool defined with @tool decorator")

✓ get_destinations tool defined with @tool decorator


In [2]:
# ============================================================
# CÁCH CŨ (deprecated — giữ lại để tham khảo, KHÔNG chạy)
# ============================================================
# agent = await provider.create_agent(
#     tools=[get_destinations],
#     name="TravelAgent",
#     instructions=(
#         "You are a helpful travel agent. Help users find their perfect vacation "
#         "destination based on their preferences. Use the get_destinations tool "
#         "to see available destinations."
#     ),
# )
#
# response = await agent.run(
#     "I'm looking for a warm beach destination. What do you recommend?"
# )
# print(response)
# ============================================================

# ============================================================
# CÁCH MỚI (agent-framework >= 1.0.0, 2026)
# ============================================================
# Thay đổi chính:
#   - Không còn dùng AzureAIProjectAgentProvider
#   - Dùng FoundryChatClient với project_endpoint (không phải connection string)
#   - Tham số deployment_name → model

#agent = FoundryChatClient(
 #   credential=credential,
 #   project_endpoint=os.getenv("FOUNDRY_PROJECT_ENDPOINT"),
 #   model=os.getenv("FOUNDRY_MODEL"),
#).as_agent(
#    name="TravelAgent",
#    instructions=(
#        "You are a helpful travel agent. Help users find their perfect vacation "
#        "destination based on their preferences. Use the get_destinations tool "
#        "to see available destinations."
#    ),
#    tools=[get_destinations],
#)

#response = await agent.run(
#    "I'm looking for a warm beach destination. What do you recommend?"
# )
#print(response.text)

import os
from azure.identity.aio import AzureCliCredential

print("Creating FoundryChatClient and Agent...")
try:
    from agent_framework_foundry import FoundryChatClient

    project_endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT") or os.getenv("FOUNDRY_PROJECT_ENDPOINT")
    if not project_endpoint:
        raise ValueError(
            "Missing project endpoint. Set AZURE_AI_PROJECT_ENDPOINT (or FOUNDRY_PROJECT_ENDPOINT) in your environment."
        )

    model_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")
    if not model_name:
        raise ValueError(
            "Missing model deployment name. Set AZURE_AI_MODEL_DEPLOYMENT_NAME in your environment."
        )
    print(f"Using model deployment: {model_name}")

    credential = AzureCliCredential()
    chat_client = FoundryChatClient(
        project_endpoint=project_endpoint,
        model=model_name,
        credential=credential,
    )
    agent = chat_client.as_agent(
        name="TravelAgent",
        instructions=(
            "You are a helpful travel agent. Help users find vacation destinations that match their preferences. "
            "Use the get_destinations tool to see available options and provide personalized recommendations."
        ),
        tools=[get_destinations],
    )
    tool_mode_enabled = True
    print("✓ Agent created successfully")
except Exception as e:
    print(f"✗ Failed to create client/agent: {type(e).__name__}: {str(e)[:200]}")
    raise

# Run the agent
print("\n" + "="*60)
print("Running Travel Agent")
print("="*60)

user_prompt = "I'm looking for a warm beach destination. What do you recommend?"
print(f"\nUser: {user_prompt}")

try:
    result = await agent.run(user_prompt)
    print(f"Agent: {result.text}")
except Exception as e:
    err_text = str(e)
    if "ApiSamplingErrorUnprocessableInput" in err_text or "Invalid parameter" in err_text:
        print("\n⚠️ This deployment does not support tool-calling parameters in this flow.")
        print("Falling back to no-tool mode so the lesson can keep running.")

        # Fallback: re-create agent without tools for model compatibility.
        agent = chat_client.as_agent(
            name="TravelAgentNoTools",
            instructions="You are a helpful travel agent. Recommend warm beach destinations based on user preferences.",
        )
        tool_mode_enabled = False
        fallback_prompt = (
            f"Use this available destination list as context: {', '.join(get_destinations())}. "
            f"User request: {user_prompt}"
        )
        result = await agent.run(fallback_prompt)
        print(f"Agent (fallback no-tool mode): {result.text}")
        print("\nTip: set AZURE_AI_MODEL_DEPLOYMENT_NAME to a tool-capable model (for example gpt-4.1-mini or gpt-4o-mini).")
    else:
        print(f"✗ Error running agent: {type(e).__name__}: {e}")
        raise

Creating FoundryChatClient and Agent...
Using model deployment: gpt-oss-120b
✓ Agent created successfully

Running Travel Agent

User: I'm looking for a warm beach destination. What do you recommend?

⚠️ This deployment does not support tool-calling parameters in this flow.
Falling back to no-tool mode so the lesson can keep running.
Agent (fallback no-tool mode): **Warm‑Beach Picks from Your List**

| Destination | Why It’s a Great Beach Spot | Best Time to Visit (warm weather) | Highlights & Activities |
|-------------|----------------------------|-----------------------------------|--------------------------|
| **Rio de Janeiro, Brazil** | Sun‑kissed, tropical coastline with iconic beaches like Copacabana and Ipanema. Temperatures stay 25‑30 °C (77‑86 °F) from December to March. | **December – March** (Brazilian summer) | • Sun‑bathing and beach volleyball<br>• Water sports: surfing, stand‑up paddle, sailing<br>• Nightlife in the Lapa district<br>• Visit the Christ the Redeemer stat

## Streaming Responses

For a more interactive experience you can **stream** the agent's response. Instead of waiting for the full reply, the agent yields text chunks as they are generated. This is especially useful in chat interfaces where you want to display output in real time.

In [3]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

I’m happy to help you plan a great getaway! My specialty is recommending warm‑weather beach destinations that match your interests. While I can give you a quick snapshot of Tokyo, the most detailed assistance I can provide is for sunny coastal spots.

Would you like some suggestions for warm beach locations—perhaps somewhere with great surf, relaxing sand, or vibrant nightlife? Let me know what you’re looking for, and I’ll tailor some perfect options for you!

## Summary

In this lesson you learned how to:

- **Create a provider** that connects to Azure AI Foundry Agent Service via `AzureAIProjectAgentProvider`.
- **Define a tool** using the `@tool` decorator so the agent can call your Python functions.
- **Run the agent** with a user message and print its response.
- **Stream responses** for real-time output.

In the next lesson we will explore agentic frameworks in more depth and learn how to give agents more powerful tools and multi-step reasoning capabilities.